1. Data Loading and *Exploration*

In [ ]:
from google.colab import files
import pandas as pd
import io
df = pd.read_csv('/content/train.csv')
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [ ]:
# Output: Total records
print(f"Total number of records: {len(df)}")

Total number of records: 7613


In [ ]:
# Output: Distribution of 0 and 1
df['target'].value_counts()

,count
target,
0,4342
1,3271


## 2. Text Normalization and **Preprocessing**

In [ ]:
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
def preprocess_text(text):
    text = str(text).lower()
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    cleaned = [w for w in tokens if w not in string.punctuation and w not in stop_words]
    return " ".join(cleaned)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


***Verify***

In [ ]:
df['cleaned_text'] = df['text'].apply(preprocess_text)
print(f"Original: {df['text'].iloc[0]}")
print(f"Cleaned:  {df['cleaned_text'].iloc[0]}")

Original: Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
Cleaned:  deeds reason earthquake may allah forgive us


3. Feature Extraction and **Ambiguity**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# a) Convert cleaned_text into Bag of Words (X)
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['cleaned_text'])
# Output: Matrix shape
print(f"Feature Matrix Shape: {X.shape}")

Feature Matrix Shape: (7613, 21804)


## **Data** **Split**

In [ ]:
from sklearn.model_selection import train_test_split
# b) Split into training (80%) and testing (20%)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=202520262)
# Output: Training and Testing sizes
print(f"Training set: {X_train.shape[0]} | Testing set: {X_test.shape[0]}")

Training set: 6090 | Testing set: 1523


# **Model Training and Evaluation**

In [ ]:
from sklearn.naive_bayes import MultinomialNB
# a) Train the model
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
# Output: Confirmation
print("Model trained successfully.")

Model trained successfully.


# **Accuracy**

In [ ]:
from sklearn.metrics import accuracy_score
# b) Predict labels
y_pred = nb_model.predict(X_test)
# Output: Accuracy Score
acc = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {acc:.4f}")

Model Accuracy on Test Set: 0.8024


In [ ]:
#Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=["Actual 0", "Actual 1"],columns=["Predicted 0", "Predicted 1"])
print("Confusion Matrix:")
print(cm_df)

Confusion Matrix:
          Predicted 0  Predicted 1
Actual 0          757          131
Actual 1          170          465


In [ ]:
# Extract recall values for comparison
from sklearn.metrics import classification_report
report = classification_report(y_test, y_pred, output_dict=True)
recall_class_0 = report['0']['recall']
recall_class_1 = report['1']['recall']
print(f"Recall for Class 0 (Not Disaster): {recall_class_0:.2f}")
print(f"Recall for Class 1 (Disaster): {recall_class_1:.2f}")
print("\nComparison:")
print("The model shows higher recall for Class 0 (Non-Disaster) compared to Class 1 (Disaster). This means it is better at detecting non-disaster tweets, but it misses some real disaster tweets, which is critical in emergency situations.")

Recall for Class 0 (Not Disaster): 0.85
Recall for Class 1 (Disaster): 0.73

Comparison:
The model shows higher recall for Class 0 (Non-Disaster) compared to Class 1 (Disaster). This means it is better at detecting non-disaster tweets, but it misses some real disaster tweets, which is critical in emergency situations.


In [ ]:
#False Negatives
# c) Identify False Negatives (Class 1 predicted as 0)
fn_count = cm[1, 0]
# Output: FN count
print("Confusion Matrix is indexed as [Actual, Predicted]")
print(f"False Negatives (Actual=1, Predicted=0): {fn_count}")

Confusion Matrix is indexed as [Actual, Predicted]
False Negatives (Actual=1, Predicted=0): 170
